# INFWIDE Final Notebook (Built from Your Working Reimplementation)

This notebook follows the same pattern that worked in your uploaded notebook:

- install missing dependencies explicitly
- place the dataset inside `INFWIDE/dataset/NightShot`
- patch the `scipy.finfo` issue
- avoid the SSIM crash during training by using `metrics=[]`
- use smaller settings that worked in Colab:
  - `batch_size=4`
  - `patch_size=64`
  - `num_workers=2`

It also includes an optional kernel-noise extension section for:
- baseline
- mild kernel noise (`std=0.001`)
- medium kernel noise (`std=0.002`)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = "/content/drive/MyDrive/NightShot"
print("Drive dataset:", DRIVE_DATA)

Mounted at /content/drive
Drive dataset: /content/drive/MyDrive/NightShot


In [2]:
%cd /content
!rm -rf INFWIDE
!git clone https://github.com/zhihongz/INFWIDE.git
%cd /content/INFWIDE

/content
Cloning into 'INFWIDE'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 164 (delta 21), reused 35 (delta 15), pack-reused 116 (from 1)
Receiving objects: 100% (164/164), 45.03 MiB | 27.01 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/INFWIDE


In [3]:
!python -m pip install --upgrade pip setuptools wheel
!pip install -q hydra-core hydra-colorlog colorlog pytorch-msssim
!pip install -q -r requirements.txt || true

import hydra, torch, skimage, pytorch_msssim
print("hydra:", hydra.__version__)
print("torch:", torch.__version__)
print("skimage:", skimage.__version__)
print("cuda available:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 79.3 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
ERROR: Could not find a version that satisfies the requirement opencv (from versions: none)
ERROR: No matching distribution found for opencv
hydra: 1.3.2
torch: 2.10.0+cu128
skimage: 0.25.2
cuda available: True


## Put dataset where the repo expects it

In [4]:
!mkdir -p /content/INFWIDE/dataset
!rm -rf /content/INFWIDE/dataset/NightShot
!ln -s "$DRIVE_DATA" /content/INFWIDE/dataset/NightShot

!ls -lah /content/INFWIDE/dataset
!ls -lah /content/INFWIDE/dataset/NightShot

total 12K
drwxr-xr-x 3 root root 4.0K Apr 14 21:17 .
drwxr-xr-x 8 root root 4.0K Apr 14 21:17 ..
lrwxrwxrwx 1 root root   32 Apr 14 21:17 NightShot -> /content/drive/MyDrive/NightShot
drwxr-xr-x 4 root root 4.0K Apr 14 21:17 NightShot_demo
lrwxrwxrwx 1 root root 32 Apr 14 21:17 /content/INFWIDE/dataset/NightShot -> /content/drive/MyDrive/NightShot


In [5]:
!find /content/INFWIDE/dataset/NightShot -maxdepth 3 -type d | sort

## Patch the `scipy.finfo` bug

In [6]:
!grep -n "scipy.finfo" -n /content/INFWIDE/srcs/utils/utils_deblur_kair.py || true
!sed -i 's/scipy\.finfo/np\.finfo/g' /content/INFWIDE/srcs/utils/utils_deblur_kair.py
!grep -n "np.finfo" -n /content/INFWIDE/srcs/utils/utils_deblur_kair.py || true

525:    h[h < scipy.finfo(float).eps * h.max()] = 0
541:    h[h < scipy.finfo(float).eps * h.max()] = 0
198:    # or np.finfo().eps
525:    h[h < np.finfo(float).eps * h.max()] = 0
541:    h[h < np.finfo(float).eps * h.max()] = 0


## Baseline training

These settings match the successful pattern from your uploaded notebook.

In [17]:
!python /content/INFWIDE/train.py \
  exp_name=baseline \
  data_loader.img_dir=/content/INFWIDE/dataset/NightShot/train_dataset \
  trainer.epochs=5 \
  data_loader.batch_size=4 \
  data_loader.patch_size=64 \
  data_loader.num_workers=2 \
  metrics=[]

/content/INFWIDE/train.py:21: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path='conf', config_name='infwide_train')
/usr/local/lib/python3.12/dist-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
 data_loader:
  _target_: srcs.data_loader.infwide_data_loaders.get_data_loaders
  img_dir: /content/INFWIDE/dataset/NightShot/train_dataset
  batch_size: 4
  patch_size: 64
  tform_op: all
  noise_type: camera
  noise_params:
    sigma_beta:
    - 0.01
    - 0.03
    sigma_r:
    - 0.5
    - 4
    nd_factor:
    - 2
    - 8
    kc:
    - 4
    - 16
  motion_len:
  - 13
  - 35
  status: train
  shuffle: true
  num_workers: 2
  validation_split: 0.05
  pin_memory:

## Find the latest checkpoint after baseline training

In [20]:
!find /content/INFWIDE/outputs -type f -name "model_best.pth" | sort

/content/INFWIDE/outputs/baseline/train/2026-04-14_21-17-48/checkpoints/model_best.pth
/content/INFWIDE/outputs/baseline/train/2026-04-14_21-32-48/checkpoints/model_best.pth
/content/INFWIDE/outputs/infwide_kn_mild/train/2026-04-14_21-52-34/checkpoints/model_best.pth


In [21]:
import glob, os
ckpts = sorted(glob.glob("/content/INFWIDE/outputs/**/model_best.pth", recursive=True))
BASELINE_CKPT = ckpts[-1] if ckpts else ""
print("BASELINE_CKPT =", BASELINE_CKPT)

BASELINE_CKPT = /content/INFWIDE/outputs/infwide_kn_mild/train/2026-04-14_21-52-34/checkpoints/model_best.pth


## Optional: kernel-noise extension patch

Run this only after baseline training works.

This patch:
- adds `perturb_kernel`
- updates dataset classes to accept kernel-noise args
- updates `get_data_loaders()` to accept and forward them
- uses the true kernel for image formation and a perturbed kernel for deblurring

In [22]:
from pathlib import Path

p = Path("/content/INFWIDE/srcs/data_loader/infwide_data_loaders.py")
txt = p.read_text()

if "def perturb_kernel(" not in txt:
    txt = txt.replace(
        "import cv2\n",
        """import cv2
import numpy as np

def perturb_kernel(psf, noise_std=0.0):
    if noise_std <= 0:
        return psf.astype(np.float32)
    noisy = psf.astype(np.float32) + np.random.normal(
        0.0, noise_std, size=psf.shape
    ).astype(np.float32)
    noisy = np.clip(noisy, 0.0, None)
    s = noisy.sum()
    if s <= 1e-8:
        noisy = psf.astype(np.float32)
        s = noisy.sum()
    noisy = noisy / (s + 1e-8)
    return noisy.astype(np.float32)
"""
    )

txt = txt.replace(
    "test_mode='one2part'):",
    "test_mode='one2part', kernel_noise_std=0.0, kernel_noise_prob=0.0):"
)

txt = txt.replace(
    "self.test_mode = test_mode",
    "self.test_mode = test_mode\n        self.kernel_noise_std = kernel_noise_std\n        self.kernel_noise_prob = kernel_noise_prob"
)

txt = txt.replace(
    "psfk = psfk.astype(np.float32)/np.sum(psfk)",
    "psfk = psfk.astype(np.float32)/(np.sum(psfk) + 1e-8)"
)
txt = txt.replace(
    "psfk = psfk.astype(np.float32) / np.sum(psfk)",
    "psfk = psfk.astype(np.float32) / (np.sum(psfk) + 1e-8)"
)

old_block = """# convolve image with psf
        coded_blur_img = ndimage.filters.convolve(
            imgk, np.expand_dims(psfk, axis=2), mode='wrap').astype(np.float32)"""

new_block = """# convolve image with true psf
        psf_true = psfk.astype(np.float32)
        psf_true = psf_true / (psf_true.sum() + 1e-8)

        if np.random.rand() < self.kernel_noise_prob:
            psf_est = perturb_kernel(psf_true, self.kernel_noise_std)
        else:
            psf_est = psf_true.copy()

        coded_blur_img = ndimage.filters.convolve(
            imgk, np.expand_dims(psf_true, axis=2), mode='wrap').astype(np.float32)

        psfk = psf_est"""

txt = txt.replace(old_block, new_block)

txt = txt.replace(
    "all2CPU=True, test_mode='one2all'):",
    "all2CPU=True, test_mode='one2all', kernel_noise_std=0.0, kernel_noise_prob=0.0):"
)

txt = txt.replace(
    "test_mode=test_mode)",
    "test_mode=test_mode, kernel_noise_std=kernel_noise_std, kernel_noise_prob=kernel_noise_prob)"
)

p.write_text(txt)
print("Patched:", p)

Patched: /content/INFWIDE/srcs/data_loader/infwide_data_loaders.py


In [23]:
import traceback, importlib
try:
    import srcs.data_loader.infwide_data_loaders as dl
    importlib.reload(dl)
    print("PATCHED IMPORT OK")
    print("get_data_loaders exists:", hasattr(dl, "get_data_loaders"))
except Exception:
    traceback.print_exc()

PATCHED IMPORT OK
get_data_loaders exists: True


## Mild kernel-noise training

In [24]:
!python /content/INFWIDE/train.py \
  exp_name=infwide_kn_mild \
  data_loader.img_dir=/content/INFWIDE/dataset/NightShot/train_dataset \
  trainer.epochs=5 \
  data_loader.batch_size=4 \
  data_loader.patch_size=64 \
  data_loader.num_workers=2 \
  +data_loader.kernel_noise_std=0.001 \
  +data_loader.kernel_noise_prob=0.7 \
  metrics=[]

/content/INFWIDE/train.py:21: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path='conf', config_name='infwide_train')
/usr/local/lib/python3.12/dist-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
 data_loader:
  _target_: srcs.data_loader.infwide_data_loaders.get_data_loaders
  img_dir: /content/INFWIDE/dataset/NightShot/train_dataset
  batch_size: 4
  patch_size: 64
  tform_op: all
  noise_type: camera
  noise_params:
    sigma_beta:
    - 0.01
    - 0.03
    sigma_r:
    - 0.5
    - 4
    nd_factor:
    - 2
    - 8
    kc:
    - 4
    - 16
  motion_len:
  - 13
  - 35
  status: train
  shuffle: true
  num_workers: 2
  validation_split: 0.05
  pin_memory:

## Medium kernel-noise training

In [25]:
!python /content/INFWIDE/train.py \
  exp_name=infwide_kn_medium \
  data_loader.img_dir=/content/INFWIDE/dataset/NightShot/train_dataset \
  trainer.epochs=5 \
  data_loader.batch_size=4 \
  data_loader.patch_size=64 \
  data_loader.num_workers=2 \
  +data_loader.kernel_noise_std=0.002 \
  +data_loader.kernel_noise_prob=0.7 \
  metrics=[]

/content/INFWIDE/train.py:21: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path='conf', config_name='infwide_train')
/usr/local/lib/python3.12/dist-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
 data_loader:
  _target_: srcs.data_loader.infwide_data_loaders.get_data_loaders
  img_dir: /content/INFWIDE/dataset/NightShot/train_dataset
  batch_size: 4
  patch_size: 64
  tform_op: all
  noise_type: camera
  noise_params:
    sigma_beta:
    - 0.01
    - 0.03
    sigma_r:
    - 0.5
    - 4
    nd_factor:
    - 2
    - 8
    kc:
    - 4
    - 16
  motion_len:
  - 13
  - 35
  status: train
  shuffle: true
  num_workers: 2
  validation_split: 0.05
  pin_memory:

## List checkpoints

In [26]:
!find /content/INFWIDE/outputs -type f -name "model_best.pth" | sort

/content/INFWIDE/outputs/baseline/train/2026-04-14_21-17-48/checkpoints/model_best.pth
/content/INFWIDE/outputs/baseline/train/2026-04-14_21-32-48/checkpoints/model_best.pth
/content/INFWIDE/outputs/infwide_kn_medium/train/2026-04-14_22-21-56/checkpoints/model_best.pth
/content/INFWIDE/outputs/infwide_kn_mild/train/2026-04-14_21-52-34/checkpoints/model_best.pth
/content/INFWIDE/outputs/infwide_kn_mild/train/2026-04-14_22-02-06/checkpoints/model_best.pth


## Optional: inference / test run from a chosen checkpoint

Set `RESUME_CKPT` to one of the printed checkpoint paths before running.

In [27]:
RESUME_CKPT = BASELINE_CKPT
print("RESUME_CKPT =", RESUME_CKPT)

RESUME_CKPT = /content/INFWIDE/outputs/infwide_kn_mild/train/2026-04-14_21-52-34/checkpoints/model_best.pth


In [28]:
!python /content/INFWIDE/train.py \
  status=test \
  resume="$RESUME_CKPT" \
  data_loader.status=test \
  data_loader.img_dir=/content/INFWIDE/dataset/NightShot/test_dataset \
  data_loader.batch_size=1 \
  data_loader.num_workers=0 \
  metrics=[]

/content/INFWIDE/train.py:21: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path='conf', config_name='infwide_train')
/usr/local/lib/python3.12/dist-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
 data_loader:
  _target_: srcs.data_loader.infwide_data_loaders.get_data_loaders
  img_dir: /content/INFWIDE/dataset/NightShot/test_dataset
  batch_size: 1
  patch_size: 256
  tform_op: all
  noise_type: camera
  noise_params:
    sigma_beta:
    - 0.01
    - 0.03
    sigma_r:
    - 0.5
    - 4
    nd_factor:
    - 2
    - 8
    kc:
    - 4
    - 16
  motion_len:
  - 13
  - 35
  status: test
  shuffle: true
  num_workers: 0
  validation_split: 0.05
  pin_memory: 

## Notes

- The TensorFlow / cuDNN warnings are not the main issue.
- The reason this notebook uses `metrics=[]` is that your logs showed SSIM crashing inside training on Colab with the current `skimage` setup. fileciteturn4file0
- This notebook follows the training pattern from the notebook you said already reimplemented successfully, then adds the kernel-noise section on top.

In [34]:
import os
import re
import cv2
import glob
import torch
import numpy as np
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# =========================
# SPEED SETTINGS
# =========================
MAX_SAMPLES_PER_BLUR = 10   # change to 5, 10, 20 as needed
USE_BLURS = ["kc4", "kc8", "kc16"]   # or only ["kc4"] for even faster result

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
print("Max samples per blur:", MAX_SAMPLES_PER_BLUR)

# =========================
# PATHS
# =========================
HU_ROOT = "/content/INFWIDE/dataset/NightShot/test_dataset/Hu_dataset_exp"
GT_DIR = os.path.join(HU_ROOT, "gt")
KERNEL_DIR = os.path.join(HU_ROOT, "kernel")

BLUR_DIRS = {
    "kc4": os.path.join(HU_ROOT, "blur_kc4"),
    "kc8": os.path.join(HU_ROOT, "blur_kc8"),
    "kc16": os.path.join(HU_ROOT, "blur_kc16"),
}

# =========================
# CHECKPOINTS
# =========================
ckpts = sorted(glob.glob("/content/INFWIDE/outputs/**/model_best.pth", recursive=True))

print("Found checkpoints:")
for c in ckpts:
    print(c)

CKPT_BASELINE = [c for c in ckpts if "/baseline/" in c][-1]
CKPT_MILD = [c for c in ckpts if "infwide_kn_mild" in c][-1]
CKPT_MED = [c for c in ckpts if "infwide_kn_medium" in c][-1]

print("\nUsing:")
print("Baseline:", CKPT_BASELINE)
print("Mild:", CKPT_MILD)
print("Medium:", CKPT_MED)

# =========================
# LOAD MODEL
# =========================
from srcs.model.infwide_model import infwide

def load_model(ckpt_path):
    model = infwide(n_colors=3, input_denoise="ResUnet")
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    if isinstance(state, dict):
        if "state_dict" in state:
            model.load_state_dict(state["state_dict"], strict=False)
        elif "model_state_dict" in state:
            model.load_state_dict(state["model_state_dict"], strict=False)
        else:
            model.load_state_dict(state, strict=False)
    else:
        raise ValueError(f"Unexpected checkpoint format: {ckpt_path}")

    model = model.to(DEVICE)
    model.eval()
    return model

# =========================
# HELPERS
# =========================
def natural_key(path):
    name = os.path.basename(path)
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', name)]

def load_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return img

def load_kernel(path):
    k = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if k is None:
        return None
    k = k.astype(np.float32)
    if k.max() > 1.0:
        k = k / 255.0
    s = k.sum()
    if s <= 1e-8:
        return None
    return k / s

def to_tensor_img(x):
    return torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0).float().to(DEVICE)

def to_tensor_kernel(k):
    return torch.from_numpy(k).unsqueeze(0).unsqueeze(0).float().to(DEVICE)

# =========================
# PREPARE FILE LISTS
# =========================
gt_files = sorted(glob.glob(os.path.join(GT_DIR, "*")), key=natural_key)
kernel_files = sorted(glob.glob(os.path.join(KERNEL_DIR, "*")), key=natural_key)

print("\nGT files:", len(gt_files))
print("Kernel files:", len(kernel_files))
for k, v in BLUR_DIRS.items():
    print(f"{k} files:", len(sorted(glob.glob(os.path.join(v, '*')), key=natural_key)))

# =========================
# FAST EVALUATION
# =========================
def evaluate_model_on_folder_fast(model, blur_dir, max_samples=10):
    blur_files = sorted(glob.glob(os.path.join(blur_dir, "*")), key=natural_key)

    n = min(len(gt_files), len(kernel_files), len(blur_files), max_samples)
    if n == 0:
        raise ValueError(f"No files found in {blur_dir}")

    psnr_list = []
    ssim_list = []

    for i in range(n):
        gt = load_rgb(gt_files[i])
        blur = load_rgb(blur_files[i])
        kernel = load_kernel(kernel_files[i])

        if gt is None or blur is None or kernel is None:
            continue

        blur_t = to_tensor_img(blur)
        kernel_t = to_tensor_kernel(kernel)

        with torch.no_grad():
            pred = model(blur_t, kernel_t)
            if isinstance(pred, (list, tuple)):
                pred = pred[1]
            pred = pred.detach().cpu().numpy()[0].transpose(1, 2, 0)

        pred = np.clip(pred, 0, 1)

        if pred.shape != gt.shape:
            pred = cv2.resize(pred, (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_CUBIC)
            pred = np.clip(pred, 0, 1)

        psnr_list.append(psnr(gt, pred, data_range=1.0))
        ssim_list.append(ssim(gt, pred, channel_axis=2, data_range=1.0))

    return {
        "matched": len(psnr_list),
        "psnr": float(np.mean(psnr_list)) if psnr_list else None,
        "ssim": float(np.mean(ssim_list)) if ssim_list else None,
    }

# =========================
# RUN FAST COMPARISON
# =========================
print("\nLoading models...\n")
models = {
    "Baseline": load_model(CKPT_BASELINE),
    "Kernel Noise 0.001": load_model(CKPT_MILD),
    "Kernel Noise 0.002": load_model(CKPT_MED),
}

rows = []

for blur_name in USE_BLURS:
    blur_dir = BLUR_DIRS[blur_name]
    print(f"\nEvaluating on {blur_name} -> {blur_dir}")
    for model_name, model in models.items():
        result = evaluate_model_on_folder_fast(model, blur_dir, MAX_SAMPLES_PER_BLUR)
        print(model_name, result)
        rows.append([
            blur_name,
            model_name,
            result["matched"],
            result["psnr"],
            result["ssim"],
        ])

df = pd.DataFrame(rows, columns=["BlurLevel", "Model", "MatchedPairs", "PSNR", "SSIM"])

print("\nFast comparison table:\n")
display(df)

summary = (
    df.groupby("Model")[["PSNR", "SSIM"]]
      .mean()
      .reset_index()
      .sort_values("PSNR", ascending=False)
      .reset_index(drop=True)
)

print("\nAverage fast comparison:\n")
display(summary)

Using device: cuda
Max samples per blur: 10
Found checkpoints:
/content/INFWIDE/outputs/baseline/train/2026-04-14_21-17-48/checkpoints/model_best.pth
/content/INFWIDE/outputs/baseline/train/2026-04-14_21-32-48/checkpoints/model_best.pth
/content/INFWIDE/outputs/infwide_kn_medium/train/2026-04-14_22-21-56/checkpoints/model_best.pth
/content/INFWIDE/outputs/infwide_kn_mild/train/2026-04-14_21-52-34/checkpoints/model_best.pth
/content/INFWIDE/outputs/infwide_kn_mild/train/2026-04-14_22-02-06/checkpoints/model_best.pth

Using:
Baseline: /content/INFWIDE/outputs/baseline/train/2026-04-14_21-32-48/checkpoints/model_best.pth
Mild: /content/INFWIDE/outputs/infwide_kn_mild/train/2026-04-14_22-02-06/checkpoints/model_best.pth
Medium: /content/INFWIDE/outputs/infwide_kn_medium/train/2026-04-14_22-21-56/checkpoints/model_best.pth

GT files: 154
Kernel files: 154
kc4 files: 154
kc8 files: 154
kc16 files: 154

Loading models...


Evaluating on kc4 -> /content/INFWIDE/dataset/NightShot/test_dataset/H

,BlurLevel,Model,MatchedPairs,PSNR,SSIM
0,kc4,Baseline,10,23.207816,0.611250
1,kc4,Kernel Noise 0.001,10,23.290599,0.641422
2,kc4,Kernel Noise 0.002,10,23.132701,0.641196
3,kc8,Baseline,10,22.674602,0.481930
4,kc8,Kernel Noise 0.001,10,22.846980,0.512980
5,kc8,Kernel Noise 0.002,10,22.699113,0.515617
6,kc16,Baseline,10,20.971716,0.310370
7,kc16,Kernel Noise 0.001,10,21.405276,0.332274
8,kc16,Kernel Noise 0.002,10,21.368753,0.336453



Average fast comparison:



,Model,PSNR,SSIM
0,Kernel Noise 0.001,22.514285,0.495559
1,Kernel Noise 0.002,22.400189,0.497755
2,Baseline,22.284711,0.467850
